<a href="https://colab.research.google.com/github/mironovromandenisovich-creator/tetstst/blob/main/notebooks/comfyui_colab_with_manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
# ==========================================
# 📦 0. УСТАНОВКА БАЗОВЫХ ПАКЕТОВ (ДО ИМПОРТОВ)
# ==========================================
!pip install -q GitPython sageattention
!apt-get install -y -qq aria2 > /dev/null 2>&1

# Скачиваем и устанавливаем официальный бинарник Cloudflared
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import os
import re
import time
import socket
import threading
import subprocess

# ==========================================
# ⚙️ 1. ТОКЕНЫ И НАСТРОЙКИ
# ==========================================
CIVITAI_TOKEN = "27950b59d0ab3d466f24b4f798c7197c"
PORT = 8188

def add_token(url, token):
    sep = "&" if "?" in url else "?"
    return f"{url}{sep}token={token}"

# ==========================================
# 🚀 2. ПОДГОТОВКА СИСТЕМЫ И ПЛАГИНОВ
# ==========================================
print("🚀 [1/5] Проверка ядра ComfyUI и кастомных нод...")

# ComfyUI Core
if not os.path.exists("/content/ComfyUI/main.py"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
else:
    %cd /content/ComfyUI
    !git pull

# ComfyUI-Manager
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
else:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !git pull

# Установка зависимостей ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# ==========================================
# 📁 3. СОЗДАНИЕ СТРУКТУРЫ ПАПОК
# ==========================================
print("\n📁 [2/5] Создание рабочих директорий...")
os.makedirs("/content/ComfyUI/models/loras", exist_ok=True)
os.makedirs("/content/ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/content/ComfyUI/models/upscale_models", exist_ok=True)

# ==========================================
# 📥 4. СКАЧИВАНИЕ МОДЕЛЕЙ И LORA
# ==========================================
print("\n📥 [3/5] Загрузка моделей и LoRA...")

# --- Б. Базовые энкодеры для оставшихся моделей картинок (HuggingFace) ---
print("  -> Проверяем Qwen3-VL и VAE...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_bf16.safetensors" \
  -d /content/ComfyUI/models/text_encoders/ -o qwen3vl_4b_bf16.safetensors

!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/vae/qwen_image_vae.safetensors" \
  -d /content/ComfyUI/models/vae/ -o qwen_image_vae.safetensors

# --- В. Апскейлер RealESRGAN ---
print("  -> Проверяем апскейлер RealESRGAN...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/spaces/Marne/Real-ESRGAN/resolve/main/RealESRGAN_x4plus.pth" \
  -d /content/ComfyUI/models/upscale_models/ -o RealESRGAN_x4plus.pth

# --- Г. Все 4 LoRA с Civitai ---
print("  -> Скачиваем LoRA с Civitai...")
loras = [
    "https://civitai.com/api/download/models/3090634?fileId=2970148",  # Krea2-realism-V2
    "https://civitai.com/api/download/models/3071904?fileId=2950820",  # HMNSFW
    "https://civitai.com/api/download/models/3290120?fileId=3174557",  # KNP LoRA
    "https://civitai.com/api/download/models/3147117?fileId=3027612"   # SNOFS Uncensored
]
for url in loras:
    auth_url = add_token(url, CIVITAI_TOKEN)
    !aria2c --content-disposition=true --console-log-level=warn -c -x 16 -s 16 -k 1M -d /content/ComfyUI/models/loras "{auth_url}"

# --- Д. Оставшиеся отборные модели картинок с Civitai ---
print("  -> Скачиваем модели картинок (Redcraft, Dark Beast, Anime v2, 3064584)...")
models = [
    "https://civitai.com/api/download/models/3139241?fileId=3259640",  # Redcraft Hybrid
    "https://civitai.com/api/download/models/3365025?fileId=3255345",  # Dark Beast 24GB
    "https://civitai.com/api/download/models/3357716?fileId=3246945",  # Krea 2 Anime v2.0
    "https://civitai.com/api/download/models/3064584?fileId=2943406"   # Модель 3064584
]
for url in models:
    auth_url = add_token(url, CIVITAI_TOKEN)
    !aria2c --content-disposition=true --console-log-level=warn -c -x 16 -s 16 -k 1M -d /content/ComfyUI/models/diffusion_models "{auth_url}"

# Симлинки в папку checkpoints (чтобы все модели были видны везде)
!ln -sf /content/ComfyUI/models/diffusion_models/* /content/ComfyUI/models/checkpoints/ 2>/dev/null || true

# ==========================================
# 🌐 5. ЗАПУСК CLOUDFLARE И COMFYUI
# ==========================================
print("\n🌐 [4/5] Настройка туннеля Cloudflare...")
!pkill -f "main.py"
!pkill -f "cloudflared"

def wait_for_port_and_launch(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\n" + "=" * 65)
    print("✅ СЕРВЕР COMFYUI УСПЕШНО ЗАПУЩЕН! ПОДКЛЮЧАЕМ CLOUDFLARE...")

    log_path = "/content/cloudflared.log"
    if os.path.exists(log_path):
        os.remove(log_path)

    # КЛЮЧЕВОЙ МОМЕНТ: флаг --protocol http2 переводит соединение на TCP
    cmd = f"cloudflared tunnel --url http://127.0.0.1:{port} --protocol http2 > {log_path} 2>&1"
    subprocess.Popen(cmd, shell=True)

    url = None
    for _ in range(40):
        time.sleep(1)
        if os.path.exists(log_path):
            with open(log_path, "r") as f:
                content = f.read()
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if match:
                    url = match.group(0)
                    break

    if url:
        print(f"👉 ВАША ССЫЛКА НА ИНТЕРФЕЙС: {url}")
    else:
        print("⚠️ Не удалось получить ссылку автоматически. Проверьте лог: !cat /content/cloudflared.log")
    print("=" * 65 + "\n")

# Запуск потока ожидания
threading.Thread(target=wait_for_port_and_launch, daemon=True, args=(PORT,)).start()

print("\n🚀 [5/5] Запуск ComfyUI...")
%cd /content/ComfyUI
# КЛЮЧЕВОЙ МОМЕНТ: флаги --listen и --enable-cors-header
!python main.py --listen 127.0.0.1 --port 8188 --enable-cors-header --dont-print-server

curl: (23) Failure writing output to destination
🚀 [1/5] Проверка ядра ComfyUI и кастомных нод...
/content/ComfyUI
Already up to date.
/content/ComfyUI/custom_nodes/ComfyUI-Manager
Already up to date.
/content/ComfyUI

📁 [2/5] Создание рабочих директорий...

📥 [3/5] Загрузка моделей и LoRA...
  -> Проверяем Qwen3-VL и VAE...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
aa8c20|OK  |       0B/s|/content/ComfyUI/models/text_encoders//qwen3vl_4b_bf16.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
e345a6|OK  |       0B/s|/content/ComfyUI/models/vae//qwen_image_vae.safetensors

Status Legend:
(OK):download completed.
  -> Проверяем апскейлер RealESRGAN...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=========================================